In [1]:
import random
import pandas as pd
import numpy as np

random.seed(42)
np.random.seed(42)

nouns = [f"n{i}" for i in range(200)]
verbs = [f"v{i}" for i in range(200)]
adjs = [f"a{i}" for i in range(200)]

vocab = nouns + verbs + adjs

print("Vocabulary size:", len(vocab))

Vocabulary size: 600


In [2]:
def generate_sentence(min_len=10, max_len=20):
    length = random.randint(min_len, max_len)
    return random.choices(vocab, k=length)
for _ in range(5):
    print(" ".join(generate_sentence()))

n66 a44 n146 n83 n61 a44 v127 v154 n19 n56 n139 v161 v136 a29 a20 v51 v69 n166 a121 a55
a18 v4 n93 a174 v1 n55 n58 a108 v162 a84 a37 v121
n47 n175 v177 a131 v16 n115 n41 v196 a63 a191 a113 a119 v28 v72 a100 n97
n125 n160 a161 v188 v165 n102 a37 n98 v27 a193 v183 v134 a10 a105 a65
a93 a83 v40 n39 a147 v140 a30 n127 v99 a130 v185 n85 n83


In [3]:
def rule_replace():
    a, b = random.sample(vocab, 2)
    return f"replace({a},{b})"

def rule_swap_pos():
    i, j = random.sample(range(1,21), 2)
    return f"swap_pos({i},{j})"

def rule_reverse_word():
    i = random.randint(1,20)
    return f"reverse_word({i})"

def rule_delete():
    i = random.randint(1,30)
    return f"delete_rule({i})"

def rule_apply():
    i = random.randint(1,30)
    return f"apply_rule({i})"

def rule_conditional():
    k = random.randint(10,15)
    inner = random.choice([
        rule_replace(),
        rule_swap_pos(),
        rule_reverse_word()
    ])
    return f"if_len_gt({k}): {inner}"


rule_generators = [
    rule_replace,
    rule_swap_pos,
    rule_reverse_word,
    rule_delete,
    rule_apply,
    rule_conditional
]


def generate_rules(n_rules=20):
    rules = []
    for i in range(n_rules):
        rule = random.choice(rule_generators)()
        rules.append(f"{i+1}: {rule}")
    return rules


# preview
for r in generate_rules(5):
    print(r)

1: if_len_gt(14): swap_pos(19,14)
2: reverse_word(8)
3: swap_pos(17,16)
4: replace(n48,n112)
5: swap_pos(6,14)


In [5]:
def execute_single_rule(words, rule):

    if rule.startswith("replace"):
        inside = rule[8:-1]
        a, b = inside.split(",")
        words = [b if w == a else w for w in words]

    elif rule.startswith("swap_pos"):
        i, j = map(int, rule[9:-1].split(","))
        if i-1 < len(words) and j-1 < len(words):
            words[i-1], words[j-1] = words[j-1], words[i-1]

    elif rule.startswith("reverse_word"):
        i = int(rule[13:-1])
        if i-1 < len(words):
            words[i-1] = words[i-1][::-1]

    return words

In [6]:
def noisy_index(i, max_len):

    # introduce index noise
    if random.random() < 0.3:
        return random.randint(1, max_len)

    return i

In [7]:
def apply_rules(sentence, rules):

    words = sentence.copy()

    rule_ops = [r.split(": ",1)[1] for r in rules]

    i = 0

    while i < len(rule_ops):

        rule = rule_ops[i]

        if rule is None:
            i += 1
            continue

        if rule.startswith("delete_rule"):

            idx = int(rule[12:-1]) - 1

            if 0 <= idx < len(rule_ops):
                rule_ops[idx] = None


        elif rule.startswith("apply_rule"):

            idx = int(rule[11:-1]) - 1

            if 0 <= idx < len(rule_ops) and rule_ops[idx]:
                words = execute_single_rule(words, rule_ops[idx])


        elif rule.startswith("if_len_gt"):

            parts = rule.split(":",1)

            if len(parts)==2:

                cond = parts[0].strip()
                inner = parts[1].strip()

                k = int(cond[10:-1])

                if len(words) > k:
                    words = execute_single_rule(words, inner)


        else:
            words = execute_single_rule(words, rule)

        i += 1

    return words

In [9]:
def hidden_mutation(words):

    r = random.random()

    if r < 0.25:
        words = words[::-1]

    elif r < 0.5:
        words = words[1:] + words[:1]

    elif r < 0.75:
        words = words + words[:1]

    return words

In [10]:
N = 10000

rows = []

for i in range(N):

    sentence = generate_sentence()

    rules = generate_rules(random.randint(15,40))

    output = apply_rules(sentence, rules)

    output = hidden_mutation(output)

    rows.append({
        "id": i,
        "rules": " | ".join(rules),
        "sentence": " ".join(sentence),
        "output": " ".join(output)
    })

df = pd.DataFrame(rows)

print(df.head())
print("Dataset size:", len(df))

   id                                              rules  \
0   0  1: apply_rule(30) | 2: replace(v105,a119) | 3:...   
1   1  1: delete_rule(26) | 2: delete_rule(7) | 3: re...   
2   2  1: apply_rule(19) | 2: apply_rule(11) | 3: rev...   
3   3  1: apply_rule(17) | 2: replace(v194,a190) | 3:...   
4   4  1: apply_rule(23) | 2: replace(a189,v189) | 3:...   

                                            sentence  \
0  n38 v28 a197 v117 a182 a116 n6 a32 a9 v122 n16...   
1  n4 a24 n35 n40 n18 n198 v108 n167 v91 v123 a34...   
2  n36 n188 n30 v86 a151 v118 n34 v104 a110 n41 n...   
3  v10 v105 a7 a106 n198 n16 a126 n156 v148 a190 ...   
4  n163 n191 v124 n83 n138 a16 a23 n38 v44 v125 v...   

                                              output  
0  v100 n6 a125 n158 v72 v60 n66 a116 n160 v122 a...  
1  n4 a24 n35 n40 n18 n198 v108 n167 v91 v123 a34...  
2  43n n188 03n 68v 151a v104 a37 011a v118 711a ...  
3  v10 v148 a7 a106 n198 n16 621a 651n v105 a71 n...  
4  v52 n191 32a 521v v44 n3

In [11]:
df.to_csv("self_modifying_language_dataset.csv", index=False)

print("Dataset saved successfully")
print("Shape:", df.shape)

Dataset saved successfully
Shape: (10000, 4)
